1. What is repartition() and why to use it 
2. Does repartition() cause a shuffle and why?
3. Can repartition() increase and decrease the no of partition
4. diffrence between df.repartition(4) and df.repartition(4,"id")
5. What happend if you repartiton 10GB of data into 10,000 partition 
6. If one key has million of records will repartition(10,"customer_id") always balance the data
7. why hash partitioning create skew
8. In what situtaion would you use repartiton() 

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataSkewBroadcastJoinDemo")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "true")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

In [ ]:
df=spark.range(0,100)
print(df.rdd.getNumPartitions())

In [ ]:
df2=df.repartition(3)
print(df2.rdd.getNumPartitions())

In [ ]:
df3=df2.repartition(5)
print(df3.rdd.getNumPartitions())

In [ ]:
data=[
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve"),
    (6, "Frank"),
    (7, "Grace"),
    (8, "Henry"),
    (9, "Ivy"),
    (10, "Jack")
]

df=spark.createDataFrame(data, ["id", "name"])
df.show()

In [ ]:
df.rdd.getNumPartitions()

In [ ]:
from pyspark.sql import functions as F
df_with_partition = df.withColumn("partition_id", F.spark_partition_id())
df_with_partition.show()

In [ ]:
df_rep=df.repartition(3)

In [ ]:
df_rep.rdd.getNumPartitions()

In [ ]:
df_rep_with_partition = df_rep.withColumn("partition_id", F.spark_partition_id())
df_rep_with_partition.show()

In [ ]:
df_rep_new=df.repartition(10)
df_rep_new.rdd.getNumPartitions()


In [ ]:
df_rep_with_partition =df_rep_new.withColumn("partition_id", F.spark_partition_id())
df_rep_with_partition.show()

In [ ]:
df_rep_new.explain()

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
df.show()

In [ ]:
df_check=df.withColumn("partition_id", spark_partition_id())
df_check.show()

In [ ]:
df_check.withColumn("partition_id", spark_partition_id()).groupBy("partition_id").count().orderBy("partition_id").show()

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
df_by_id=df.repartition(5, "id")

df_by_id.withColumn("partition_id",spark_partition_id()).orderBy("id").show(50,False)

In [ ]:
df_by_id.explain()

In [ ]:
from pyspark.sql.functions import spark_partition_id
data=[
    (1,"A"),
    (1,"B"),
    (2,"C"),
    (2,"D"),
    (3,"E"),
    (4,"F"),
    (4,"G"),
]

df=spark.createDataFrame(data, ["id", "name"])
df.show()

In [ ]:
df_hash=df.repartition(5, "id")


In [ ]:
df_hash.withColumn("partition_id",spark_partition_id()).orderBy("id").show(50,False)

In [ ]:
from pyspark.sql.functions import hash,col

df=spark.createDataFrame(
    [(1,),(2,),(3,),(4,),(5,)]
)

df.withColumn("hash_value", hash(col("_1"))).show()

In [ ]:
from pyspark.sql.functions import hash,col,pmod,lit 

df2=(
    df
    .withColumn("hash_value", hash(col("_1")))
    .withColumn(
        "expected_partition",
    pmod(col("hash_value"), lit(5))
    )
)

df2.show()

In [ ]:
df.printSchema()

In [ ]:
df=spark.range(0,1_000_000)

df.rdd.getNumPartitions()

In [ ]:
df_small_partitions=df.repartition(2)
print(df_small_partitions.rdd.getNumPartitions())

In [ ]:
(df_small_partitions
.withColumn("partition_id",spark_partition_id())
.groupBy("partition_id")
.count()
.orderBy("partition_id")
.show()
)

In [ ]:
df_better=df_small_partitions.repartition(10)
print(df_better.rdd.getNumPartitions())

In [ ]:
(df_better
.withColumn("partition_id",spark_partition_id())
.groupBy("partition_id")
.count()
.orderBy("partition_id")
.show()
)

# Colasec 

In [ ]:
from pyspark.sql.functions import spark_partition_id
df=spark.range(1,21)
print("Initial partitions: ", df.rdd.getNumPartitions())

In [ ]:
df_coalesced=df.coalesce(2)
print("After coalesce: ", df_coalesced.rdd.getNumPartitions())

In [ ]:
(
    df_coalesced
    .withColumn("partition_id",spark_partition_id())
    .orderBy("partition_id","id")
    .show(50,False)
)

In [ ]:
df_coalesced.explain()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id,col
spark=(
    SparkSession.builder
    .appName("CoalesceDemo")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

In [ ]:
df=spark.range(
    start=1,
    end=1_000_001,
    numPartitions=8
)

In [ ]:
print("Initial partitions: ", df.rdd.getNumPartitions())

In [ ]:
def count_partition_rows(partition_id,rows):
    count=0
    for row in rows:
        count+=1
    yield (partition_id,count)

partition_counts=(
    df.rdd.mapPartitionsWithIndex(count_partition_rows)
    .collect()
)

partition_counts

In [ ]:
filtered_df=df.filter(
    col("id") % 20 == 0
)

In [ ]:
filtered_df.count()

In [ ]:
print(
    "Partition after filter:", filtered_df.rdd.getNumPartitions()
)

In [ ]:
def count_partition_rows(partition_id,rows):
    count=0
    for row in rows:
        count+=1
    yield (partition_id,count)

partition_counts=(
    filtered_df.rdd.mapPartitionsWithIndex(count_partition_rows)
    .collect()
)

partition_counts

In [ ]:
filtered_df.write.mode("overwrite").parquet("output/coalesced_output/normal_output/")

In [ ]:
filtered_df.coalesce(2).write.mode("overwrite").parquet("output/coalesced_output/normal_output/")

# Data Skew Practical 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import spark_partition_id,col
spark=(
    SparkSession.builder
    .appName("DataSkewDemo")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/06 12:55:21 WARN Utils: Your hostname, Anujs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.124 instead (on interface en0)
26/10/06 12:55:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/anujshahdeo/Documents/Full_Stack_AWS_Data_Engineering/full-stack-aws-data-engineering/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/06 12:55:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin

Spark Version       : 4.2.0
Application Name    : DataSkewDemo
Master              : local[*]
Default Parallelism : 8
Shuffle Partitions  : 200
Spark UI            : http://192.168.29.124:4040


In [2]:
data=(
    [("A",)]*90000 +
    [("B",)]*5000 +
    [("C",)]*3000 +
    [("D",)]*2000 
)

df=spark.createDataFrame(data, ["key"])

In [3]:
df.count()

100000

In [4]:
df.groupBy("key").count().orderBy("count", ascending=False).show()


+---+-----+
|key|count|
+---+-----+
|  A|90000|
|  B| 5000|
|  C| 3000|
|  D| 2000|
+---+-----+



In [5]:
df_skew=df.repartition(4, "key")

In [7]:
from pyspark.sql.functions import spark_partition_id,col
(
    df_skew
    .withColumn("partition_id",spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")    
    .show()
)

+------------+-----+
|partition_id|count|
+------------+-----+
|           1| 8000|
|           2|92000|
+------------+-----+



In [8]:
from pyspark.sql.functions import spark_partition_id,col
(
    df_skew
    .withColumn("partition_id",spark_partition_id())
    .groupBy("key", "partition_id")
    .count()
    .orderBy("partition_id","key")
    .show()
)

+---+------------+-----+
|key|partition_id|count|
+---+------------+-----+
|  B|           1| 5000|
|  C|           1| 3000|
|  A|           2|90000|
|  D|           2| 2000|
+---+------------+-----+



In [9]:
def count_rows_per_partition(index,iterator):
    count=sum(1 for _ in iterator)
    yield (index,count)

df_skew.rdd.mapPartitionsWithIndex(count_rows_per_partition).collect()

[(0, 0), (1, 8000), (2, 92000), (3, 0)]